# Wheat Disease Classification — MobileViT training

Trains a `MobileViT` (from `vit-pytorch`) on the Kaggle `freedomfighter1290/wheat-disease` dataset, from scratch (no pretrained weights exist for this architecture in the library), and exports the result to ONNX for `ai-vision`'s FastAPI service.

`CLASS_NAMES` below must stay in sync with `ai-vision/app/main.py` — the assertion cell fails loudly if the dataset's folders don't match.

In [15]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import classification_report

from vit_pytorch.mobile_vit import MobileViT

torch.backends.cudnn.benchmark = True

In [16]:
# Must stay in sync with CLASS_NAMES in ai-vision/app/main.py
CLASS_NAMES = [
    "Aphid", "Black Rust", "Blast", "Brown Rust",
    "Fusarium Head Blight", "Healthy Wheat", "Leaf Blight",
    "Mildew", "Mite", "Septoria", "Smut", "Stem fly",
    "Tan spot", "Yellow Rust",
]

IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
EPOCHS = 30
WARMUP_EPOCHS = 5
LR = 3e-4
WEIGHT_DECAY = 0.05
VAL_SPLIT = 0.2
SEED = 42
# Measured: NUM_WORKERS=0 makes data loading (1866ms/batch) ~10x the GPU compute cost
# (178ms/batch), dominating total training time. NUM_WORKERS=4 tested working with this
# notebook's exact ImageFolder/transforms.Compose (both fully picklable) and cuts data
# loading to ~595ms/batch. If this ever hangs in your actual kernel, drop back to 0.
NUM_WORKERS = 4

WEIGHTS_DIR = Path("app/model/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = WEIGHTS_DIR / "best_model.pt"
ONNX_PATH = WEIGHTS_DIR / "final_model.onnx"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Training on: cuda NVIDIA GeForce RTX 4050 Laptop GPU


In [17]:
def find_train_dir(root_candidates) -> Path:
    """Locate the folder whose immediate subfolders are the 14 class names, wherever the Kaggle zip extracted them, regardless of the kernel's working directory."""
    target = set(CLASS_NAMES)
    candidates = []
    for root in root_candidates:
        candidates += [root / "Wheat_Disease" / "train", root / "train", root]
    for c in candidates:
        if c.is_dir():
            subdirs = {p.name for p in c.iterdir() if p.is_dir()}
            if target.issubset(subdirs):
                return c
    raise FileNotFoundError(
        f"Could not find a folder whose subfolders match CLASS_NAMES. Checked: {candidates}"
    )


# Try both "data" (kernel cwd == ai-vision/) and "ai-vision/data" (kernel cwd == repo root)
TRAIN_DIR = find_train_dir([Path("data"), Path("ai-vision/data")])
DATA_ROOT = TRAIN_DIR.parent.parent if TRAIN_DIR.parent.name == "train" else TRAIN_DIR.parent
print("Using data from:", TRAIN_DIR)

discovered = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
assert discovered == sorted(CLASS_NAMES), (
    f"Class folders don't match CLASS_NAMES!\nFound: {discovered}\nExpected: {sorted(CLASS_NAMES)}"
)

class_counts = {}
for name in CLASS_NAMES:
    count = len(list((TRAIN_DIR / name).glob("*")))
    class_counts[name] = count
    print(f"{name}: {count} images")

Using data from: data\Wheat_Disease\train
Aphid: 773 images
Black Rust: 933 images
Blast: 517 images
Brown Rust: 3248 images
Fusarium Head Blight: 1020 images
Healthy Wheat: 2907 images
Leaf Blight: 673 images
Mildew: 884 images
Mite: 640 images
Septoria: 1140 images
Smut: 1048 images
Stem fly: 187 images
Tan spot: 616 images
Yellow Rust: 2379 images


In [18]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# RandomResizedCrop instead of a plain square Resize: this dataset mixes thumbnails and
# full-resolution photos across many aspect ratios, so a plain Resize(IMAGE_SIZE) would
# squish/distort non-square images. RandomResizedCrop crops a region then resizes it,
# avoiding that distortion while also acting as scale augmentation.
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Eval: resize-then-center-crop (deterministic, no squish) instead of a plain square resize.
_eval_resize = int(IMAGE_SIZE[0] * 1.14)
eval_transform = transforms.Compose([
    transforms.Resize(_eval_resize),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# The dataset ships its own split under this exact folder name (confirmed from the
# extracted data -- NOT "val").
VAL_DIR = TRAIN_DIR.parent / "validation"

if VAL_DIR.is_dir():
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
    class_to_idx = train_dataset.class_to_idx
else:
    # Two ImageFolder instances over the same directory (one per transform), split by
    # shared indices — avoids the classic bug where random_split'ing one ImageFolder
    # and then reassigning .transform on the val Subset mutates the train Subset too.
    train_tf_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    eval_tf_dataset = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)

    n = len(train_tf_dataset)
    val_size = int(n * VAL_SPLIT)
    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(n, generator=generator).tolist()
    train_indices, val_indices = indices[val_size:], indices[:val_size]

    train_dataset = Subset(train_tf_dataset, train_indices)
    val_dataset = Subset(eval_tf_dataset, val_indices)
    class_to_idx = train_tf_dataset.class_to_idx

assert [k for k, _ in sorted(class_to_idx.items(), key=lambda kv: kv[1])] == CLASS_NAMES, (
    "ImageFolder's alphabetical class-to-index mapping doesn't match CLASS_NAMES order."
)

In [19]:
from PIL import Image
from tqdm.auto import tqdm


def ahash(path, hash_size=8):
    """Average hash: cheap perceptual fingerprint, robust to re-compression/minor edits."""
    with Image.open(path) as img:
        img = img.convert("L").resize((hash_size, hash_size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float64)
    bits = (arr > arr.mean()).flatten()
    h = 0
    for b in bits:
        h = (h << 1) | int(b)
    return h


def get_samples(ds):
    return [ds.dataset.samples[i] for i in ds.indices] if isinstance(ds, Subset) else ds.samples


# This dataset's own train/validation split turned out to have ~63% overlap (checked
# separately with the same aHash approach): 52% of validation images are exact
# duplicates of a training image, another ~11% are near-duplicates. Reported validation
# accuracy would be badly inflated without removing those. Train's internal duplication
# is left alone -- redundant compute isn't a correctness problem the way train/val leakage is.
train_samples = get_samples(train_dataset)
val_samples = get_samples(val_dataset)

train_hashes = np.array(
    [ahash(p) for p, _ in tqdm(train_samples, desc="Hashing train images")],
    dtype=np.uint64,
)
train_hash_set = set(int(h) for h in train_hashes)

clean_indices = []
for i, (path, _label) in enumerate(tqdm(val_samples, desc="Checking validation for train duplicates")):
    h = ahash(path)
    if h in train_hash_set:
        continue
    xor = train_hashes ^ np.uint64(h)
    if any(bin(int(x)).count("1") <= 4 for x in xor):
        continue
    clean_indices.append(i)

print(f"Validation: {len(val_samples)} total -> {len(clean_indices)} after removing "
      f"train duplicates/near-duplicates ({len(val_samples) - len(clean_indices)} dropped)")

val_dataset = Subset(val_dataset, clean_indices)

C:\Users\Yassin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking validation for train duplicates: 100%|██████████| 4247/4247 [01:20<00:00, 52.68it/s] 

Validation: 4247 total -> 1529 after removing train duplicates/near-duplicates (2718 dropped)


In [20]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

Train samples: 16965, Val samples: 1529


In [21]:
# mobilevit_xs configuration, per vit-pytorch's README example, with num_classes swapped to ours
model = MobileViT(
    image_size=IMAGE_SIZE,
    dims=[96, 120, 144],
    channels=[16, 32, 48, 48, 64, 64, 80, 80, 96, 96, 384],
    num_classes=len(CLASS_NAMES),
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"MobileViT params: {num_params:,}")

MobileViT params: 2,004,320


In [22]:
# Inverse-frequency class weights -- the dataset is imbalanced (~17x between the
# largest and smallest class), so an unweighted loss would bias the model toward
# the majority classes (Brown Rust, Healthy Wheat, Yellow Rust).
total_train = sum(class_counts.values())
num_classes = len(CLASS_NAMES)
class_weights = torch.tensor(
    [total_train / (num_classes * class_counts[name]) for name in CLASS_NAMES],
    dtype=torch.float32,
).to(DEVICE)
print("Class weights:", {name: round(w, 2) for name, w in zip(CLASS_NAMES, class_weights.tolist())})

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=(DEVICE.type == "cuda"))

Class weights: {'Aphid': 1.57, 'Black Rust': 1.3, 'Blast': 2.34, 'Brown Rust': 0.37, 'Fusarium Head Blight': 1.19, 'Healthy Wheat': 0.42, 'Leaf Blight': 1.8, 'Mildew': 1.37, 'Mite': 1.89, 'Septoria': 1.06, 'Smut': 1.16, 'Stem fly': 6.48, 'Tan spot': 1.97, 'Yellow Rust': 0.51}


In [23]:
import time
from tqdm.auto import tqdm


def format_duration(seconds: float) -> str:
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    return f"{h}h{m:02d}m{s:02d}s" if h else f"{m}m{s:02d}s"


def run_epoch(loader, train: bool, desc: str):
    model.train(mode=train)
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        pbar = tqdm(loader, desc=desc, leave=False)
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
            pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.4f}")
    return total_loss / total, correct / total


best_val_acc = 0.0
patience, bad_epochs = 5, 0
epoch_times = []
training_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    train_loss, train_acc = run_epoch(train_loader, train=True, desc=f"Epoch {epoch+1}/{EPOCHS} [train]")
    val_loss, val_acc = run_epoch(val_loader, train=False, desc=f"Epoch {epoch+1}/{EPOCHS} [val]")
    scheduler.step()

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    eta = avg_epoch_time * (EPOCHS - (epoch + 1))
    elapsed = time.time() - training_start

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
          f"| val_loss {val_loss:.4f} acc {val_acc:.4f} "
          f"| epoch {format_duration(epoch_time)} | elapsed {format_duration(elapsed)} "
          f"| ETA {format_duration(eta)}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        bad_epochs = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  -> new best val accuracy ({best_val_acc:.4f}), checkpoint saved")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break

print(f"Training finished in {format_duration(time.time() - training_start)}. Best val accuracy: {best_val_acc:.4f}")

Epoch 1/30 | train_loss 2.3083 acc 0.2719 | val_loss 2.0870 acc 0.3466 | epoch 3m39s | elapsed 3m39s | ETA 1h45m53s
  -> new best val accuracy (0.3466), checkpoint saved


Epoch 2/30 | train_loss 2.0049 acc 0.3892 | val_loss 1.9348 acc 0.3996 | epoch 3m08s | elapsed 6m47s | ETA 1h35m00s
  -> new best val accuracy (0.3996), checkpoint saved


Epoch 3/30 | train_loss 1.7883 acc 0.4503 | val_loss 1.5715 acc 0.4787 | epoch 3m10s | elapsed 9m57s | ETA 1h29m34s
  -> new best val accuracy (0.4787), checkpoint saved


Epoch 4/30 | train_loss 1.6660 acc 0.4744 | val_loss 1.4696 acc 0.5154 | epoch 3m15s | elapsed 13m12s | ETA 1h25m51s
  -> new best val accuracy (0.5154), checkpoint saved


Epoch 5/30 | train_loss 1.5550 acc 0.5041 | val_loss 1.4439 acc 0.5252 | epoch 3m12s | elapsed 16m24s | ETA 1h22m04s
  -> new best val accuracy (0.5252), checkpoint saved


...

Epoch 25/30 | train_loss 0.5725 acc 0.7901 | val_loss 0.9076 acc 0.7188 | epoch 3m47s | elapsed 1h27m47s | ETA 17m33s


Epoch 26/30 | train_loss 0.5445 acc 0.7994 | val_loss 0.9490 acc 0.7188 | epoch 3m59s | elapsed 1h31m47s | ETA 14m07s


Epoch 27/30 | train_loss 0.5259 acc 0.8039 | val_loss 0.9301 acc 0.7181 | epoch 3m49s | elapsed 1h35m37s | ETA 10m37s


Epoch 28/30 | train_loss 0.5185 acc 0.8073 | val_loss 0.9072 acc 0.7220 | epoch 3m39s | elapsed 1h39m17s | ETA 7m05s


Epoch 29/30 | train_loss 0.5049 acc 0.8093 | val_loss 0.9273 acc 0.7227 | epoch 3m37s | elapsed 1h42m54s | ETA 3m32s
Early stopping at epoch 29 (no improvement for 5 epochs)
Training finished in 1h42m54s. Best val accuracy: 0.7253


In [24]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images.to(DEVICE))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

                      precision    recall  f1-score   support

               Aphid       0.65      0.63      0.64       128
          Black Rust       0.58      0.67      0.62        84
               Blast       0.73      0.72      0.73        46
          Brown Rust       0.79      0.75      0.77       245
Fusarium Head Blight       0.93      0.96      0.95        72
       Healthy Wheat       0.95      0.79      0.86       257
         Leaf Blight       0.39      0.62      0.48        79
              Mildew       0.64      0.81      0.72        69
                Mite       0.64      0.62      0.63       110
            Septoria       0.40      0.57      0.47        40
                Smut       0.00      0.00      0.00         0
            Stem fly       0.33      0.56      0.42        16
            Tan spot       0.52      0.43      0.47        91
         Yellow Rust       0.93      0.82      0.87       292

            accuracy                           0.72      1529
      

C:\Users\Yassin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Yassin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Yassin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py

In [25]:
model.eval()
dummy_input = torch.randn(1, 3, *IMAGE_SIZE, device=DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
print(f"Exported ONNX model to {ONNX_PATH}")

# Serving-side preprocessing must match this exactly — saved so main.py's rewrite doesn't have to guess.
preprocess_config = {
    "image_size": list(IMAGE_SIZE),
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
    "class_names": CLASS_NAMES,
}
with open(WEIGHTS_DIR / "preprocess_config.json", "w") as f:
    json.dump(preprocess_config, f, indent=2)
print("Saved preprocess_config.json")

Exported ONNX model to app\model\weights\final_model.onnx
Saved preprocess_config.json


In [26]:
import onnxruntime as ort

session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
sample_images, sample_labels = next(iter(val_loader))
sample = sample_images[:1].numpy()

onnx_logits = session.run(None, {"input": sample})[0]
with torch.no_grad():
    torch_logits = model(sample_images[:1].to(DEVICE)).cpu().numpy()

print("Max abs diff between PyTorch and ONNX outputs:", np.abs(onnx_logits - torch_logits).max())
print("ONNX predicted class:", CLASS_NAMES[onnx_logits.argmax()])
print("True label:", CLASS_NAMES[sample_labels[0].item()])

Max abs diff between PyTorch and ONNX outputs: 0.0019791126
ONNX predicted class: Aphid
True label: Aphid


## Next steps

- `final_model.onnx`, `preprocess_config.json` now sit in `ai-vision/app/model/weights/`.
- `ai-vision/app/main.py` still expects a `.keras` file — it needs to be rewritten to load `final_model.onnx` via `onnxruntime.InferenceSession`, resize/normalize with the same `image_size`/`mean`/`std` saved above, and apply softmax to the raw logits ONNX returns (this model has no softmax layer).